# Customer Support Copilot — Thin Orchestrator

This notebook is intentionally tiny. It does **not** contain any business logic — every function is imported from the `models/`, `retrieval/`, `training/`, and `utils/` packages.

Sections:
1. Set up paths and seed
2. Load + split dataset, write chunks to disk
3. Optional: train (bi-encoder then DoRA)
4. Run evaluation under `config/baseline.yaml` and `config/full.yaml`
5. Print baseline-vs-full comparison

In [ ]:
import os, sys, json, random
from pathlib import Path

ROOT = Path('..').resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from utils.config import load_config
print('cwd:', os.getcwd())

## 1) Chunk the raw MultiDoc2Dial dump into train/test splits
Writes `data/processed/chunks.json`, `data/splits/train.json`, `data/splits/test.json`.

In [ ]:
from collections import defaultdict
import re

cfg = load_config('config/config.yaml')
raw_path = cfg['data']['raw_path']
if not os.path.exists(raw_path):
    print(f'⚠️  {raw_path} not found — place MultiDoc2Dial raw dump here before running.')
else:
    with open(raw_path) as f:
        docs = json.load(f)
    size, overlap = cfg['data']['chunk_size'], cfg['data']['chunk_overlap']
    chunks = []
    for d in docs:
        text = d.get('text') or d.get('doc_text') or ''
        if not text:
            continue
        words = re.sub(r'\s+', ' ', text).split()
        step  = size - overlap
        for i in range(0, len(words), step):
            t = ' '.join(words[i:i+size])
            if len(t.split()) < 20:
                continue
            chunks.append({'chunk_id': f"{d['doc_id']}__{i//step}",
                           'doc_id':   d['doc_id'],
                           'text':     t})
    random.Random(cfg.get('seed', 42)).shuffle(chunks)
    doc_ids = list({c['doc_id'] for c in chunks})
    random.Random(cfg.get('seed', 42)).shuffle(doc_ids)
    cut = int(len(doc_ids) * cfg['data']['train_ratio'])
    train_docs, test_docs = set(doc_ids[:cut]), set(doc_ids[cut:])
    train = [c for c in chunks if c['doc_id'] in train_docs]
    test  = [c for c in chunks if c['doc_id'] in test_docs]
    os.makedirs('data/processed', exist_ok=True)
    os.makedirs('data/splits', exist_ok=True)
    json.dump(chunks, open(cfg['data']['processed_path'], 'w'))
    json.dump(train,  open(cfg['data']['train_split'], 'w'))
    json.dump(test,   open(cfg['data']['test_split'], 'w'))
    print(f'✅ {len(chunks)} chunks  train={len(train)}  test={len(test)}')

## 2) (optional) Train: A = bi-encoder, C+D = DoRA + DPO
Skip if you already have checkpoints.

In [ ]:
# !python -m training.train --stage biencoder --config config/full.yaml
# !python -m training.train --stage dora      --config config/full.yaml

## 3) Evaluate baseline, then full, then compare
Each run writes JSON to `experiments/results/`. `--compare` prints the delta table.

In [ ]:
!python -m training.evaluation --config config/baseline.yaml
!python -m training.evaluation --config config/full.yaml
!python -m training.evaluation --compare

## 4) Interactive demo
Either CLI (`python -m demo.cli --config config/full.yaml`) or FastAPI (`uvicorn demo.app:app --port 8000`).